In [28]:
# Cell 1 — Imports and database connection

import psycopg2
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from numpy.linalg import lstsq
import warnings
warnings.filterwarnings('ignore')

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print("Connection established.")
print(f"psycopg2 version: {psycopg2.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Connection established.
psycopg2 version: 2.9.12 (dt dec pq3 ext lo64)
pandas version: 2.3.3
numpy version: 2.0.2


In [29]:
# Cell 2 — Compute per-game per-team metrics from raw.plays

RUSH_TYPES    = ('Rush', 'Rushing Touchdown')
PASS_TYPES    = ('Pass Reception', 'Pass Incompletion', 'Passing Touchdown',
                 'Sack', 'Pass Completion', 'Pass Interception Return')
SCRIMMAGE_TYPES = RUSH_TYPES + PASS_TYPES

rush_in  = "(" + ", ".join(f"'{t}'" for t in RUSH_TYPES)  + ")"
pass_in  = "(" + ", ".join(f"'{t}'" for t in PASS_TYPES)  + ")"
scrim_in = "(" + ", ".join(f"'{t}'" for t in SCRIMMAGE_TYPES) + ")"

query = f"""
WITH scrimmage AS (
    SELECT
        game_id, season, week, offense, defense,
        play_type, yards_gained, yards_to_goal,
        down, distance, ppa, scoring, drive_id,
        clock_minutes, clock_seconds,
        CASE
            WHEN down = 1 AND yards_gained >= distance * 0.5 THEN 1
            WHEN down = 2 AND yards_gained >= distance * 0.7 THEN 1
            WHEN down IN (3,4) AND yards_gained >= distance   THEN 1
            ELSE 0
        END AS success,
        CASE
            WHEN down = 1 THEN 1
            WHEN down = 2 AND distance <= 8 THEN 1
            WHEN down IN (3,4) AND distance <= 5 THEN 1
            ELSE 0
        END AS is_std_down,
        CASE
            WHEN down = 2 AND distance > 8 THEN 1
            WHEN down IN (3,4) AND distance > 5 THEN 1
            ELSE 0
        END AS is_pass_down,
        CASE WHEN play_type IN {rush_in} THEN 1 ELSE 0 END AS is_rush,
        CASE WHEN play_type IN {pass_in} THEN 1 ELSE 0 END AS is_pass,
        CASE WHEN play_type = 'Sack'     THEN 1 ELSE 0 END AS is_sack,
        CASE
            WHEN play_type IN {rush_in} AND yards_gained < 0   THEN yards_gained * 1.2
            WHEN play_type IN {rush_in} AND yards_gained <= 4   THEN yards_gained * 1.0
            WHEN play_type IN {rush_in} AND yards_gained <= 10  THEN 4 + (yards_gained - 4) * 0.5
            WHEN play_type IN {rush_in} AND yards_gained > 10   THEN 7.0
            ELSE NULL
        END AS line_yards,
        CASE WHEN yards_to_goal <= 10 THEN 1 ELSE 0 END AS is_redzone
    FROM raw.plays
    WHERE season_type = 'regular'
      AND play_type IN {scrim_in}
      AND down IS NOT NULL
      AND distance IS NOT NULL
),

drive_clock AS (
    SELECT game_id, season, offense, drive_id,
           MAX(clock_minutes * 60 + clock_seconds) - MIN(clock_minutes * 60 + clock_seconds) AS drive_seconds
    FROM raw.plays
    WHERE season_type = 'regular'
      AND play_type NOT IN ('Kickoff','Timeout','End of Half','End of Game',
                            'End Period','End of Regulation','placeholder')
      AND clock_minutes IS NOT NULL AND clock_seconds IS NOT NULL
    GROUP BY game_id, season, offense, drive_id
),

top_agg AS (
    SELECT game_id, season, offense,
           SUM(drive_seconds) AS time_of_possession
    FROM drive_clock
    GROUP BY game_id, season, offense
),

scoring_drives AS (
    SELECT game_id, offense, drive_id,
           MAX(CASE WHEN scoring = TRUE THEN 1 ELSE 0 END) AS is_scoring_drive
    FROM raw.plays
    WHERE season_type = 'regular'
      AND play_type IN {scrim_in}
    GROUP BY game_id, offense, drive_id
),

pts_opp AS (
    SELECT game_id, offense,
           SUM(is_scoring_drive)::float / NULLIF(COUNT(DISTINCT drive_id), 0) AS off_pts_per_opportunity
    FROM scoring_drives
    GROUP BY game_id, offense
),

off_metrics AS (
    SELECT
        game_id, season, week, offense AS team_name,
        AVG(CASE WHEN is_rush=1 THEN success END)                              AS off_success_rate_rush,
        AVG(CASE WHEN is_pass=1 THEN success END)                              AS off_success_rate_pass,
        AVG(CASE WHEN is_std_down=1  THEN success END)                         AS off_success_rate_std_downs,
        AVG(CASE WHEN is_pass_down=1 THEN success END)                         AS off_success_rate_pass_downs,
        AVG(CASE WHEN yards_gained >= 20 THEN 1.0 ELSE 0.0 END)               AS off_explosive_rate_20,
        AVG(CASE WHEN yards_gained >= 10 THEN 1.0 ELSE 0.0 END)               AS off_explosive_rate_10,
        AVG(CASE WHEN is_rush=1 AND yards_gained <= 0 THEN 1.0
                 WHEN is_rush=1 THEN 0.0 END)                                  AS off_stuff_rate,
        AVG(CASE WHEN is_rush=1 THEN line_yards END)                           AS off_line_yards_per_rush,
        SUM(CASE WHEN is_sack=1 THEN 1 ELSE 0 END)::float / NULLIF(SUM(is_pass),0)
                                                                               AS off_sack_rate_allowed,
        AVG(CASE WHEN is_rush=1 THEN ppa END)                                  AS off_epa_rush,
        AVG(CASE WHEN is_pass=1 THEN ppa END)                                  AS off_epa_pass,
        AVG(CASE WHEN is_redzone=1 THEN success END)                           AS off_success_rate_redzone,
        AVG(CASE WHEN is_redzone=1 THEN ppa END)                               AS off_epa_redzone,
        AVG(CASE WHEN is_std_down=1  AND is_rush=1 THEN 1.0
                 WHEN is_std_down=1  AND is_pass=1 THEN 0.0 END)              AS rush_rate_std_downs,
        AVG(CASE WHEN is_pass_down=1 AND is_rush=1 THEN 1.0
                 WHEN is_pass_down=1 AND is_pass=1 THEN 0.0 END)              AS rush_rate_pass_downs
    FROM scrimmage
    GROUP BY game_id, season, week, offense
),

def_metrics AS (
    SELECT
        game_id, season, week, defense AS team_name,
        AVG(CASE WHEN is_rush=1 THEN success END)                              AS def_success_rate_rush,
        AVG(CASE WHEN is_pass=1 THEN success END)                              AS def_success_rate_pass,
        AVG(CASE WHEN is_std_down=1  THEN success END)                         AS def_success_rate_std_downs,
        AVG(CASE WHEN is_pass_down=1 THEN success END)                         AS def_success_rate_pass_downs,
        AVG(CASE WHEN yards_gained >= 20 THEN 1.0 ELSE 0.0 END)               AS def_explosive_rate_20_allowed,
        AVG(CASE WHEN yards_gained >= 10 THEN 1.0 ELSE 0.0 END)               AS def_explosive_rate_10_allowed,
        AVG(CASE WHEN is_rush=1 AND yards_gained <= 0 THEN 1.0
                 WHEN is_rush=1 THEN 0.0 END)                                  AS def_stuff_rate_allowed,
        AVG(CASE WHEN is_rush=1 THEN line_yards END)                           AS def_line_yards_per_rush_allowed,
        SUM(CASE WHEN is_sack=1 THEN 1 ELSE 0 END)::float / NULLIF(SUM(is_pass),0)
                                                                               AS def_sack_rate,
        AVG(CASE WHEN is_rush=1 THEN ppa END)                                  AS def_epa_rush_allowed,
        AVG(CASE WHEN is_pass=1 THEN ppa END)                                  AS def_epa_pass_allowed,
        AVG(CASE WHEN is_redzone=1 THEN success END)                           AS def_success_rate_redzone_allowed,
        AVG(CASE WHEN is_redzone=1 THEN ppa END)                               AS def_epa_redzone_allowed
    FROM scrimmage
    GROUP BY game_id, season, week, defense
)

SELECT
    o.game_id, o.season, o.week, o.team_name,
    o.off_success_rate_rush, o.off_success_rate_pass,
    o.off_success_rate_std_downs, o.off_success_rate_pass_downs,
    o.off_explosive_rate_20, o.off_explosive_rate_10,
    o.off_stuff_rate, o.off_line_yards_per_rush, o.off_sack_rate_allowed,
    o.off_epa_rush, o.off_epa_pass,
    o.off_success_rate_redzone, o.off_epa_redzone,
    o.rush_rate_std_downs, o.rush_rate_pass_downs,
    d.def_success_rate_rush, d.def_success_rate_pass,
    d.def_success_rate_std_downs, d.def_success_rate_pass_downs,
    d.def_explosive_rate_20_allowed, d.def_explosive_rate_10_allowed,
    d.def_stuff_rate_allowed, d.def_line_yards_per_rush_allowed, d.def_sack_rate,
    d.def_epa_rush_allowed, d.def_epa_pass_allowed,
    d.def_success_rate_redzone_allowed, d.def_epa_redzone_allowed,
    t.time_of_possession,
    p.off_pts_per_opportunity
FROM off_metrics o
LEFT JOIN def_metrics d  ON o.game_id = d.game_id AND o.team_name = d.team_name
LEFT JOIN top_agg t      ON o.game_id = t.game_id AND o.team_name = t.offense
LEFT JOIN pts_opp p      ON o.game_id = p.game_id AND o.team_name = p.offense
ORDER BY o.season, o.week, o.game_id, o.team_name
"""

print("Running per-game metric query...")
game_metrics = pd.read_sql(query, conn)

numeric_cols = [c for c in game_metrics.columns if c not in ['game_id','team_name','season','week']]
game_metrics[numeric_cols] = game_metrics[numeric_cols].apply(pd.to_numeric, errors='coerce').astype('float64')

print(f"Shape: {game_metrics.shape}")
print(f"Seasons: {sorted(game_metrics['season'].unique())}")
print(f"Null counts (top 10):\n{game_metrics.isnull().sum().sort_values(ascending=False).head(10)}")

Running per-game metric query...
Shape: (12013, 34)
Seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Null counts (top 10):
def_epa_redzone_allowed             1516
off_epa_redzone                     1516
def_success_rate_redzone_allowed    1451
off_success_rate_redzone            1451
def_epa_pass_allowed                  12
def_epa_rush_allowed                  12
off_epa_rush                           8
def_success_rate_pass_downs            7
off_epa_pass                           7
def_line_yards_per_rush_allowed        6
dtype: int64


In [30]:
# Cell 3 — Fix def_pts_per_opportunity_allowed via pandas self-join

pts_lookup = game_metrics[['game_id','team_name','off_pts_per_opportunity']].copy()
pts_lookup = pts_lookup.rename(columns={
    'team_name':               'opponent',
    'off_pts_per_opportunity': 'def_pts_per_opportunity_allowed'
})

game_metrics = game_metrics.merge(pts_lookup, on='game_id', how='left')
game_metrics = game_metrics[game_metrics['team_name'] != game_metrics['opponent']].copy()
game_metrics = game_metrics.drop(columns=['opponent'])

game_counts = game_metrics.groupby('game_id').size()
print(f"Shape after fix: {game_metrics.shape}")
print(f"Games with != 2 rows: {(game_counts != 2).sum()}")
print(f"Nulls in def_pts_per_opportunity_allowed: {game_metrics['def_pts_per_opportunity_allowed'].isnull().sum()}")

Shape after fix: (12008, 35)
Games with != 2 rows: 0
Nulls in def_pts_per_opportunity_allowed: 0


In [31]:
# Cell 4 — Build 3-game rolling windows

METRIC_COLS = [
    'off_success_rate_rush', 'off_success_rate_pass',
    'off_success_rate_std_downs', 'off_success_rate_pass_downs',
    'def_success_rate_rush', 'def_success_rate_pass',
    'def_success_rate_std_downs', 'def_success_rate_pass_downs',
    'off_explosive_rate_20', 'off_explosive_rate_10',
    'def_explosive_rate_20_allowed', 'def_explosive_rate_10_allowed',
    'off_stuff_rate', 'off_line_yards_per_rush', 'off_sack_rate_allowed',
    'def_stuff_rate_allowed', 'def_line_yards_per_rush_allowed', 'def_sack_rate',
    'off_pts_per_opportunity', 'def_pts_per_opportunity_allowed',
    'off_epa_rush', 'off_epa_pass',
    'def_epa_rush_allowed', 'def_epa_pass_allowed',
    'off_success_rate_redzone', 'def_success_rate_redzone_allowed',
    'off_epa_redzone', 'def_epa_redzone_allowed',
    'time_of_possession',
    'rush_rate_std_downs', 'rush_rate_pass_downs',
]

game_metrics = game_metrics.sort_values(['team_name','season','week']).reset_index(drop=True)

def compute_rolling(df, cols, window=3):
    rolled_parts = []
    for (team, season), grp in df.groupby(['team_name','season'], sort=False):
        grp = grp.sort_values('week').copy()
        rolled = grp[cols].shift(1).rolling(window=window, min_periods=1).mean()
        rolled.columns = [f'last3_{c}' for c in cols]
        combined = pd.concat([grp[['game_id','season','week','team_name']], rolled], axis=1)
        rolled_parts.append(combined)
    return pd.concat(rolled_parts, ignore_index=True)

print("Computing rolling windows...")
rolled = compute_rolling(game_metrics, METRIC_COLS)

game_metrics_rolled = game_metrics.merge(
    rolled, on=['game_id','season','week','team_name'], how='left'
)

print(f"Shape: {game_metrics_rolled.shape}")
print(f"Rolling cols added: {len([c for c in game_metrics_rolled.columns if c.startswith('last3_')])}")
print("Done.")

Computing rolling windows...
Shape: (12008, 66)
Rolling cols added: 31
Done.


In [32]:
# Cell 5 — Load control variables from authoritative tables
#
# AUTHORITATIVE SOURCES (from candidate_features.csv):
#   conference, sp_rating        -> int_team_season_features (team_name + season)
#   close_game_epa_per_play etc  -> int_game_team_features   (game_id + team_name)
#   conference_game flag         -> raw.games                (game_id only — never join on team name)
#
# KEY INSIGHT FROM PRIOR EDA:
#   int_game_team_features.team_name is the authoritative FBS team name string.
#   raw.games.home_team / away_team do NOT match int_team_season_features.team_name.
#   All team-level joins must go through int_game_team_features.team_name.

# Step 1: Load int_game_team_features — this is the FBS team name universe
epa_df = pd.read_sql("""
    SELECT
        game_id, team_name, season, week,
        opponent,
        points_scored, points_allowed,
        close_game_epa_per_play,
        close_game_def_epa_per_play
    FROM int.int_game_team_features
""", conn)

epa_num = ['points_scored','points_allowed',
           'close_game_epa_per_play','close_game_def_epa_per_play']
epa_df[epa_num] = epa_df[epa_num].apply(pd.to_numeric, errors='coerce').astype('float64')

# Step 2: Load conference + SP+ from int_team_season_features
conf_sp_df = pd.read_sql("""
    SELECT team_name, season, conference, sp_rating
    FROM int.int_team_season_features
""", conn)
conf_sp_df['sp_rating'] = pd.to_numeric(conf_sp_df['sp_rating'], errors='coerce').astype('float64')

# Step 3: Get conference_game flag from raw.games by game_id only
conf_game_df = pd.read_sql("""
    SELECT id AS game_id, conference_game
    FROM raw.games
    WHERE season_type = 'regular'
""", conn)

# Step 4: Attach conference + SP+ to epa_df via team_name + season
epa_df = epa_df.merge(conf_sp_df[['team_name','season','conference','sp_rating']],
                      on=['team_name','season'], how='left')

# Step 5: Attach conference_game flag via game_id
epa_df = epa_df.merge(conf_game_df, on='game_id', how='left')

print(f"EPA df shape: {epa_df.shape}")
print(f"Null conference:    {epa_df['conference'].isnull().sum()}")
print(f"Null sp_rating:     {epa_df['sp_rating'].isnull().sum()}")
print(f"Null conf_game:     {epa_df['conference_game'].isnull().sum()}")
print(f"\nConferences:\n{epa_df['conference'].value_counts().to_string()}")

EPA df shape: (29472, 12)
Null conference:    22991
Null sp_rating:     22991
Null conf_game:     0

Conferences:
conference
Big Ten              776
ACC                  749
SEC                  728
Big 12               680
Sun Belt             677
American Athletic    644
Mid-American         596
Mountain West        586
Conference USA       513
Pac-12               340
FBS Independents     192


In [33]:
# Cell 6 — Build matchup dataset
#
# Strategy: use int_game_team_features as the spine.
# Each game appears twice (once per team). Use team + opponent columns
# to construct home/away matchup rows. Filter to FBS conference games only.
# Independents excluded: they have conference=FBS Independents, no conference_game=True rows.

# Filter to FBS conference games only
conf_epa = epa_df[
    (epa_df['conference_game'] == True) &
    (epa_df['conference'].notna()) &
    (epa_df['conference'] != 'FBS Independents')
].copy()

print(f"FBS conference game-team rows: {len(conf_epa)}")
print(f"Conference distribution:\n{conf_epa['conference'].value_counts().to_string()}")

# Each game has 2 rows: team and opponent. Build matchup by self-joining on game_id.
# Designate the row with is_home logic using points_scored/allowed symmetry:
# team A: points_scored = home_points, points_allowed = away_points
# team B: points_scored = away_points, points_allowed = home_points
# We need to identify which team is home. Use raw.games home_team for this.

home_away_df = pd.read_sql("""
    SELECT id AS game_id, home_team, away_team
    FROM raw.games
    WHERE season_type = 'regular'
""", conn)

# Tag each row in conf_epa as home or away
conf_epa = conf_epa.merge(home_away_df, on='game_id', how='left')
conf_epa['is_home'] = conf_epa['team_name'] == conf_epa['home_team']

# Separate into home and away perspectives
home_df = conf_epa[conf_epa['is_home'] == True].copy()
away_df = conf_epa[conf_epa['is_home'] == False].copy()

print(f"\nHome rows: {len(home_df)}")
print(f"Away rows: {len(away_df)}")

FBS conference game-team rows: 4310
Conference distribution:
conference
Big Ten              582
Big 12               510
ACC                  500
SEC                  486
Sun Belt             454
American Athletic    434
Mid-American         398
Mountain West        378
Conference USA       342
Pac-12               226

Home rows: 2155
Away rows: 2155


In [34]:
# Cell 7 — Build matchups and merge rolled metrics

ROLL_COLS = [c for c in game_metrics_rolled.columns if c.startswith('last3_')]

# Build matchup spine from home_df — columns already have home_team and away_team
matchups = home_df[[
    'game_id', 'season', 'week',
    'home_team', 'away_team',
    'points_scored', 'points_allowed',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'sp_rating', 'conference'
]].copy().rename(columns={
    'points_scored':               'home_points',
    'points_allowed':              'away_points',
    'close_game_epa_per_play':     'home_cg_epa',
    'close_game_def_epa_per_play': 'home_cg_def_epa',
    'sp_rating':                   'home_sp',
})

# Away EPA controls from away_df — already has home_team and away_team
away_controls = away_df[[
    'game_id', 'away_team',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'sp_rating'
]].copy().rename(columns={
    'close_game_epa_per_play':     'away_cg_epa',
    'close_game_def_epa_per_play': 'away_cg_def_epa',
    'sp_rating':                   'away_sp',
})

matchups = matchups.merge(away_controls, on=['game_id','away_team'], how='left')

# Rolled metrics — join on game_id + home_team/away_team + season
home_rolled = game_metrics_rolled[['game_id','team_name','season'] + ROLL_COLS].copy()
home_rolled = home_rolled.rename(columns={
    'team_name': 'home_team',
    **{c: f'home_{c}' for c in ROLL_COLS}
})

away_rolled = game_metrics_rolled[['game_id','team_name','season'] + ROLL_COLS].copy()
away_rolled = away_rolled.rename(columns={
    'team_name': 'away_team',
    **{c: f'away_{c}' for c in ROLL_COLS}
})

matchups = matchups.merge(home_rolled, on=['game_id','home_team','season'], how='left')
matchups = matchups.merge(away_rolled, on=['game_id','away_team','season'], how='left')

# Outcomes and control deltas
matchups['point_differential']   = matchups['home_points']  - matchups['away_points']
matchups['total_points']         = matchups['home_points']  + matchups['away_points']
matchups['close_game_epa_delta'] = matchups['home_cg_epa']  - matchups['away_cg_def_epa']
matchups['sp_rating_delta']      = matchups['home_sp']       - matchups['away_sp']

# Tier assignment
P4_CONFERENCES = {"ACC","Big 12","Big Ten","SEC"}
matchups['tier'] = matchups['conference'].apply(
    lambda c: 'P4' if c in P4_CONFERENCES else 'G5'
)

print(f"Matchup rows: {len(matchups)}")
print(f"Null point_differential:   {matchups['point_differential'].isnull().sum()}")
print(f"Null close_game_epa_delta: {matchups['close_game_epa_delta'].isnull().sum()}")
print(f"Null sp_rating_delta:      {matchups['sp_rating_delta'].isnull().sum()}")
print(f"\nConference distribution:\n{matchups['conference'].value_counts().to_string()}")
print(f"\nTier distribution:\n{matchups['tier'].value_counts().to_string()}")

Matchup rows: 2155
Null point_differential:   0
Null close_game_epa_delta: 3
Null sp_rating_delta:      0

Conference distribution:
conference
Big Ten              291
Big 12               255
ACC                  250
SEC                  243
Sun Belt             227
American Athletic    217
Mid-American         199
Mountain West        189
Conference USA       171
Pac-12               113

Tier distribution:
tier
G5    1116
P4    1039


In [35]:
# Cell 8 — Construct matchup deltas for all style/tempo features

DELTA_PAIRS = [
    ('off_success_rate_rush',       'def_success_rate_rush'),
    ('off_success_rate_pass',       'def_success_rate_pass'),
    ('off_success_rate_std_downs',  'def_success_rate_std_downs'),
    ('off_success_rate_pass_downs', 'def_success_rate_pass_downs'),
    ('off_explosive_rate_20',       'def_explosive_rate_20_allowed'),
    ('off_explosive_rate_10',       'def_explosive_rate_10_allowed'),
    ('off_stuff_rate',              'def_stuff_rate_allowed'),
    ('off_line_yards_per_rush',     'def_line_yards_per_rush_allowed'),
    ('off_sack_rate_allowed',       'def_sack_rate'),
    ('off_pts_per_opportunity',     'def_pts_per_opportunity_allowed'),
    ('off_epa_rush',                'def_epa_rush_allowed'),
    ('off_epa_pass',                'def_epa_pass_allowed'),
    ('off_success_rate_redzone',    'def_success_rate_redzone_allowed'),
    ('off_epa_redzone',             'def_epa_redzone_allowed'),
]

TEMPO_SOLO = ['time_of_possession','rush_rate_std_downs','rush_rate_pass_downs']

for off_col, def_col in DELTA_PAIRS:
    matchups[f'delta_{off_col}'] = (
        matchups[f'home_last3_{off_col}'] - matchups[f'away_last3_{def_col}']
    )

for col in TEMPO_SOLO:
    matchups[f'delta_{col}'] = (
        matchups[f'home_last3_{col}'] - matchups[f'away_last3_{col}']
    )

DELTA_COLS = [f'delta_{o}' for o, _ in DELTA_PAIRS] + [f'delta_{c}' for c in TEMPO_SOLO]

print(f"Delta columns created: {len(DELTA_COLS)}")
null_rates = matchups[DELTA_COLS].isnull().mean().sort_values(ascending=False)
print(f"\nDelta null rates (top 10):\n{null_rates.head(10).to_string()}")

Delta columns created: 17

Delta null rates (top 10):
delta_off_epa_redzone             0.016241
delta_off_success_rate_redzone    0.015777
delta_off_success_rate_rush       0.007889
delta_off_pts_per_opportunity     0.007889
delta_rush_rate_std_downs         0.007889
delta_time_of_possession          0.007889
delta_off_epa_pass                0.007889
delta_off_epa_rush                0.007889
delta_off_sack_rate_allowed       0.007889
delta_off_success_rate_pass       0.007889


In [36]:
# Cell 9 — Partial r utility functions

def partial_r(df, feature, outcome, controls):
    cols = [feature, outcome] + controls
    sub  = df[cols].dropna()
    n    = len(sub)
    if n < 30:
        return np.nan, n

    def residualize(y, X):
        X_ = np.column_stack([np.ones(len(X)), X])
        coef, _, _, _ = lstsq(X_, y, rcond=None)
        return y - X_ @ coef

    X_ctrl        = sub[controls].values
    resid_feat    = residualize(sub[feature].values, X_ctrl)
    resid_outcome = residualize(sub[outcome].values, X_ctrl)

    if resid_feat.std() == 0 or resid_outcome.std() == 0:
        return np.nan, n

    r, _ = pearsonr(resid_feat, resid_outcome)
    return round(r, 4), n

def signal_label(r):
    if np.isnan(r):  return 'insufficient data'
    a = abs(r)
    if a >= 0.15:    return 'anchor candidate'
    if a >= 0.08:    return 'supporting candidate'
    return 'redundant / drop'

CONTROLS    = ['close_game_epa_delta', 'sp_rating_delta']
OUTCOMES    = ['point_differential', 'total_points']
CONFERENCES = [
    'ACC','American Athletic','Big 12','Big Ten',
    'Conference USA','Mid-American','Mountain West',
    'Pac-12','SEC','Sun Belt'
]

print("Utility functions defined.")

Utility functions defined.


In [37]:
# Cell 10 — Run partial r: full population, P4, G5, each conference

def run_all_partial_r(df, delta_cols, controls, outcomes):
    records = []
    for delta in delta_cols:
        row = {'feature_delta': delta}
        for outcome in outcomes:
            prefix = 'spread' if outcome == 'point_differential' else 'ou'

            r_full, n_full = partial_r(df, delta, outcome, controls)
            row[f'{prefix}_partial_r_full'] = r_full
            row[f'{prefix}_n_full']         = n_full
            row[f'{prefix}_signal']         = signal_label(r_full)

            r_p4, _ = partial_r(df[df['tier'] == 'P4'], delta, outcome, controls)
            row[f'{prefix}_partial_r_p4'] = r_p4

            r_g5, _ = partial_r(df[df['tier'] == 'G5'], delta, outcome, controls)
            row[f'{prefix}_partial_r_g5'] = r_g5

            conf_rs     = {}
            conf_labels = {}
            for conf in CONFERENCES:
                r_c, _ = partial_r(df[df['conference'] == conf], delta, outcome, controls)
                conf_rs[conf]     = r_c
                conf_labels[conf] = signal_label(r_c)

            row[f'{prefix}_conf_r']      = conf_rs
            row[f'{prefix}_conf_signal'] = conf_labels

        records.append(row)
    return pd.DataFrame(records)

print("Running partial r tests — this may take a minute...")
results_df = run_all_partial_r(matchups, DELTA_COLS, CONTROLS, OUTCOMES)

print(f"Results shape: {results_df.shape}")
print("\nTop spread signals (full population):")
print(
    results_df[['feature_delta','spread_partial_r_full','spread_signal']]
    .sort_values('spread_partial_r_full', key=abs, ascending=False)
    .to_string(index=False)
)
print("\nTop O/U signals (full population):")
print(
    results_df[['feature_delta','ou_partial_r_full','ou_signal']]
    .sort_values('ou_partial_r_full', key=abs, ascending=False)
    .to_string(index=False)
)

Running partial r tests — this may take a minute...
Results shape: (17, 15)

Top spread signals (full population):
                    feature_delta  spread_partial_r_full    spread_signal
               delta_off_epa_pass                 0.0507 redundant / drop
      delta_off_success_rate_pass                 0.0416 redundant / drop
    delta_off_pts_per_opportunity                 0.0325 redundant / drop
 delta_off_success_rate_std_downs                 0.0305 redundant / drop
            delta_off_epa_redzone                 0.0243 redundant / drop
        delta_rush_rate_std_downs                -0.0225 redundant / drop
             delta_off_stuff_rate                 0.0195 redundant / drop
      delta_off_success_rate_rush                -0.0188 redundant / drop
       delta_rush_rate_pass_downs                -0.0180 redundant / drop
               delta_off_epa_rush                -0.0177 redundant / drop
   delta_off_success_rate_redzone                 0.0173 redundant / dr

In [38]:
# Cell 10a — Moneyline variance signal test
#
# Variance signal: does the delta predict the RESIDUAL VARIANCE of the outcome
# after controlling for EPA and SP+?
# Method: regress outcome on controls, compute squared residuals,
# then correlate each delta with those squared residuals.
# A positive correlation means larger delta -> larger unpredictability (spread).
# Report separately for point_differential and total_points.

from numpy.linalg import lstsq

def compute_outcome_residuals(df, outcome, controls):
    """Residualize outcome on controls, return squared residuals aligned to df index."""
    cols = [outcome] + controls
    sub  = df[cols].dropna()
    X    = np.column_stack([np.ones(len(sub)), sub[controls].values])
    coef, _, _, _ = lstsq(X, sub[outcome].values, rcond=None)
    resid = sub[outcome].values - X @ coef
    return pd.Series(resid**2, index=sub.index)

def variance_signal_r(df, feature, sq_resid):
    """Correlate feature with squared residuals. Returns (r, n)."""
    aligned = pd.concat([df[feature], sq_resid], axis=1).dropna()
    n = len(aligned)
    if n < 30:
        return np.nan, n
    r, _ = pearsonr(aligned.iloc[:,0], aligned.iloc[:,1])
    return round(r, 4), n

# Compute squared residuals for spread and O/U on full population
sq_resid_spread = compute_outcome_residuals(matchups, 'point_differential', CONTROLS)
sq_resid_ou     = compute_outcome_residuals(matchups, 'total_points',       CONTROLS)

variance_records = []
for delta in DELTA_COLS:
    r_sp, n_sp = variance_signal_r(matchups, delta, sq_resid_spread)
    r_ou, n_ou = variance_signal_r(matchups, delta, sq_resid_ou)
    variance_records.append({
        'feature_delta':          delta,
        'variance_r_spread':      r_sp,
        'variance_signal_spread': abs(r_sp) >= 0.08 if not np.isnan(r_sp) else False,
        'variance_n_spread':      n_sp,
        'variance_r_ou':          r_ou,
        'variance_signal_ou':     abs(r_ou) >= 0.08 if not np.isnan(r_ou) else False,
        'variance_n_ou':          n_ou,
    })

variance_df = pd.DataFrame(variance_records)

print("Moneyline variance signal results:")
print(variance_df[['feature_delta','variance_r_spread','variance_signal_spread',
                    'variance_r_ou','variance_signal_ou']].to_string(index=False))

Moneyline variance signal results:
                    feature_delta  variance_r_spread  variance_signal_spread  variance_r_ou  variance_signal_ou
      delta_off_success_rate_rush             0.0291                   False         0.0024               False
      delta_off_success_rate_pass             0.0020                   False        -0.0130               False
 delta_off_success_rate_std_downs             0.0225                   False        -0.0063               False
delta_off_success_rate_pass_downs            -0.0237                   False        -0.0152               False
      delta_off_explosive_rate_20            -0.0150                   False         0.0125               False
      delta_off_explosive_rate_10            -0.0162                   False         0.0049               False
             delta_off_stuff_rate            -0.0092                   False        -0.0217               False
    delta_off_line_yards_per_rush             0.0173                 

In [39]:
# Cell 10b — Conference-stratified partial r display table
#
# Print a readable conference breakdown for every delta feature.
# Spread and O/U reported separately per the handoff requirement.

def format_conf_table(results_df, outcome_prefix):
    rows = []
    for _, row in results_df.iterrows():
        delta    = row['feature_delta']
        conf_r   = row.get(f'{outcome_prefix}_conf_r', {})
        full_r   = row[f'{outcome_prefix}_partial_r_full']
        p4_r     = row[f'{outcome_prefix}_partial_r_p4']
        g5_r     = row[f'{outcome_prefix}_partial_r_g5']
        signal   = row[f'{outcome_prefix}_signal']

        conf_str = "  ".join([f"{k[:6]}={v:.3f}" if not np.isnan(v) else f"{k[:6]}=nan"
                               for k, v in conf_r.items()])
        rows.append({
            'feature':  delta,
            'full_r':   full_r,
            'p4_r':     p4_r,
            'g5_r':     g5_r,
            'signal':   signal,
            'by_conf':  conf_str,
        })
    return pd.DataFrame(rows)

print("=" * 100)
print("SPREAD PARTIAL R — CONFERENCE STRATIFICATION")
print("=" * 100)
spread_conf = format_conf_table(results_df, 'spread')
for _, row in spread_conf.iterrows():
    print(f"\n{row['feature']}")
    print(f"  Full={row['full_r']:.4f}  P4={row['p4_r']:.4f}  G5={row['g5_r']:.4f}  [{row['signal']}]")
    print(f"  {row['by_conf']}")

print("\n" + "=" * 100)
print("O/U PARTIAL R — CONFERENCE STRATIFICATION")
print("=" * 100)
ou_conf = format_conf_table(results_df, 'ou')
for _, row in ou_conf.iterrows():
    print(f"\n{row['feature']}")
    print(f"  Full={row['full_r']:.4f}  P4={row['p4_r']:.4f}  G5={row['g5_r']:.4f}  [{row['signal']}]")
    print(f"  {row['by_conf']}")

SPREAD PARTIAL R — CONFERENCE STRATIFICATION

delta_off_success_rate_rush
  Full=-0.0188  P4=-0.0375  G5=-0.0028  [redundant / drop]
  ACC=-0.038  Americ=-0.023  Big 12=-0.042  Big Te=-0.057  Confer=-0.043  Mid-Am=-0.092  Mounta=0.067  Pac-12=0.142  SEC=0.005  Sun Be=-0.049

delta_off_success_rate_pass
  Full=0.0416  P4=0.0647  G5=0.0212  [redundant / drop]
  ACC=0.036  Americ=0.043  Big 12=0.084  Big Te=0.007  Confer=-0.076  Mid-Am=0.024  Mounta=0.077  Pac-12=0.003  SEC=0.161  Sun Be=-0.007

delta_off_success_rate_std_downs
  Full=0.0305  P4=0.0298  G5=0.0305  [redundant / drop]
  ACC=0.037  Americ=0.036  Big 12=0.019  Big Te=0.003  Confer=-0.058  Mid-Am=-0.033  Mounta=0.125  Pac-12=0.066  SEC=0.089  Sun Be=-0.001

delta_off_success_rate_pass_downs
  Full=-0.0056  P4=-0.0002  G5=-0.0116  [redundant / drop]
  ACC=-0.073  Americ=-0.001  Big 12=0.016  Big Te=-0.016  Confer=-0.066  Mid-Am=-0.043  Mounta=-0.004  Pac-12=0.079  SEC=0.077  Sun Be=-0.023

delta_off_explosive_rate_20
  Full=0.0

In [40]:
# Cell 11 — Within-season trajectory by arc bin

def game_arc_bin(week):
    if week == 1:      return '1_conf_game_1'
    elif week <= 4:    return '2_games_2_4'
    elif week <= 8:    return '3_games_5_8'
    else:              return '4_games_9_plus'

matchups['arc_bin'] = matchups['week'].apply(game_arc_bin)

arc_records = []
for delta in DELTA_COLS:
    for arc, grp in matchups.groupby('arc_bin'):
        r_sp, n = partial_r(grp, delta, 'point_differential', CONTROLS)
        r_ou, _ = partial_r(grp, delta, 'total_points',       CONTROLS)
        arc_records.append({
            'feature_delta': delta,
            'arc_bin':       arc,
            'spread_r':      r_sp,
            'ou_r':          r_ou,
            'n':             n
        })

arc_df = pd.DataFrame(arc_records)

def trajectory_verdict(rs):
    rs = [r for r in rs if not np.isnan(r)]
    if len(rs) < 2:             return 'insufficient'
    if rs[-1] > rs[0] + 0.05:  return 'strengthens'
    if rs[-1] < rs[0] - 0.05:  return 'weakens'
    return 'stable'

traj_verdicts = {}
for delta in DELTA_COLS:
    sub = arc_df[arc_df['feature_delta'] == delta].sort_values('arc_bin')
    traj_verdicts[delta] = trajectory_verdict(sub['spread_r'].tolist())

print("Trajectory verdicts:")
print(pd.Series(traj_verdicts).value_counts().to_string())

Trajectory verdicts:
weakens        9
stable         7
strengthens    1


In [41]:
# Cell 12 — YoY stability

season_avg = game_metrics.groupby(['team_name','season'])[METRIC_COLS].mean().reset_index()
seasons    = sorted(season_avg['season'].unique())

yoy_records = []
for col in METRIC_COLS:
    rs = []
    for yr in seasons[:-1]:
        yr_n  = season_avg[season_avg['season'] == yr    ][['team_name',col]].rename(columns={col:'yr_n'})
        yr_n1 = season_avg[season_avg['season'] == yr + 1][['team_name',col]].rename(columns={col:'yr_n1'})
        merged = yr_n.merge(yr_n1, on='team_name').dropna()
        if len(merged) >= 20:
            r, _ = pearsonr(merged['yr_n'], merged['yr_n1'])
            rs.append(r)
    avg_r = np.mean(rs) if rs else np.nan

    if np.isnan(avg_r):   verdict = 'insufficient'
    elif avg_r >= 0.70:   verdict = 'stable'
    elif avg_r >= 0.50:   verdict = 'moderate'
    else:                 verdict = 'unstable'

    yoy_records.append({'metric': col, 'yoy_r': round(avg_r, 4), 'yoy_verdict': verdict})

yoy_df = pd.DataFrame(yoy_records).sort_values('yoy_r', ascending=False)
print("YoY stability results:")
print(yoy_df.to_string(index=False))

YoY stability results:
                          metric  yoy_r yoy_verdict
      off_success_rate_std_downs 0.5674    moderate
             rush_rate_std_downs 0.5348    moderate
 def_pts_per_opportunity_allowed 0.5263    moderate
           off_success_rate_rush 0.5238    moderate
         off_pts_per_opportunity 0.5162    moderate
           off_success_rate_pass 0.4917    unstable
                  off_stuff_rate 0.4825    unstable
            rush_rate_pass_downs 0.4785    unstable
         off_line_yards_per_rush 0.4719    unstable
           off_explosive_rate_10 0.4425    unstable
                    off_epa_rush 0.4356    unstable
           def_success_rate_rush 0.4282    unstable
      def_success_rate_std_downs 0.4197    unstable
            def_epa_rush_allowed 0.3996    unstable
          def_stuff_rate_allowed 0.3960    unstable
                    off_epa_pass 0.3955    unstable
   def_explosive_rate_10_allowed 0.3915    unstable
   def_explosive_rate_20_allowed 0.3641  

In [42]:
# Cell 13 — Assemble and write verdict CSV (includes variance signal columns)

import os

output_dir = os.path.expanduser('~/cfb-analytics/artifacts')
os.makedirs(output_dir, exist_ok=True)

metric_to_delta = {}
for off_col, def_col in DELTA_PAIRS:
    metric_to_delta[off_col] = f'delta_{off_col}'
for col in TEMPO_SOLO:
    metric_to_delta[col] = f'delta_{col}'

yoy_lookup = {}
for _, row in yoy_df.iterrows():
    delta = metric_to_delta.get(row['metric'])
    if delta and delta not in yoy_lookup:
        yoy_lookup[delta] = (row['yoy_r'], row['yoy_verdict'])

# Index variance_df by feature_delta for lookup
variance_lookup = variance_df.set_index('feature_delta')

verdict_rows = []
for _, row in results_df.iterrows():
    delta         = row['feature_delta']
    yoy_r, yoy_v  = yoy_lookup.get(delta, (np.nan, 'no mapping'))
    traj          = traj_verdicts.get(delta, 'unknown')
    var_row       = variance_lookup.loc[delta] if delta in variance_lookup.index else {}

    conf_sp_signals = [
        k for k, v in row.get('spread_conf_signal', {}).items()
        if v in ('anchor candidate', 'supporting candidate')
    ]
    conf_ou_signals = [
        k for k, v in row.get('ou_conf_signal', {}).items()
        if v in ('anchor candidate', 'supporting candidate')
    ]

    verdict_rows.append({
        'feature_delta':             delta,
        'spread_partial_r_full':     row['spread_partial_r_full'],
        'spread_signal':             row['spread_signal'],
        'ou_partial_r_full':         row['ou_partial_r_full'],
        'ou_signal':                 row['ou_signal'],
        'variance_signal_spread':    var_row.get('variance_signal_spread', False),
        'variance_r_spread':         var_row.get('variance_r_spread', np.nan),
        'variance_signal_ou':        var_row.get('variance_signal_ou', False),
        'variance_r_ou':             var_row.get('variance_r_ou', np.nan),
        'trajectory_verdict':        traj,
        'yoy_r':                     yoy_r,
        'yoy_verdict':               yoy_v,
        'conferences_spread_signal': str(conf_sp_signals),
        'conferences_ou_signal':     str(conf_ou_signals),
        'notes':                     ''
    })

verdict_df = pd.DataFrame(verdict_rows)
out_path   = os.path.join(output_dir, 'style_tempo_verdict.csv')
verdict_df.to_csv(out_path, index=False)

print(f"Verdict CSV written to: {out_path}")
print(f"Rows: {len(verdict_df)}")
print("\nFull verdict table:")
print(verdict_df[['feature_delta','spread_partial_r_full','spread_signal',
                   'ou_partial_r_full','ou_signal',
                   'variance_signal_spread','variance_signal_ou',
                   'trajectory_verdict','yoy_r','yoy_verdict']].to_string(index=False))

Verdict CSV written to: /Users/kevinjohnson/cfb-analytics/artifacts/style_tempo_verdict.csv
Rows: 17

Full verdict table:
                    feature_delta  spread_partial_r_full    spread_signal  ou_partial_r_full        ou_signal  variance_signal_spread  variance_signal_ou trajectory_verdict  yoy_r yoy_verdict
      delta_off_success_rate_rush                -0.0188 redundant / drop             0.0265 redundant / drop                   False               False            weakens 0.5238    moderate
      delta_off_success_rate_pass                 0.0416 redundant / drop             0.0463 redundant / drop                   False               False            weakens 0.4917    unstable
 delta_off_success_rate_std_downs                 0.0305 redundant / drop             0.0499 redundant / drop                   False               False            weakens 0.5674    moderate
delta_off_success_rate_pass_downs                -0.0056 redundant / drop             0.0064 redundant / drop 

In [43]:
# Cell 14 — Close DB connection

cur.close()
conn.close()
print("Connection closed. Notebook complete.")

Connection closed. Notebook complete.
